# Scrambled Topology Analysis

**Goal:** Test whether GenNet's topology matters for prediction performance.

**Approach:**
1. Use existing pure simulation data (genotypes already generated)
2. Original topology: Contiguous SNP blocks → Genes
3. Scrambled topology: Randomly reassign SNP → Gene connections
4. Train GenNet on both topologies
5. Compare performance

**Hypothesis:** 
- If topology matches causal structure → original wins
- If causal structure is random → scrambled competitive or wins

**Note:** In pure synthetic data, both topologies are arbitrary relative to causal structure, so we expect minimal difference. This is a proof-of-concept for the methodology.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil
from types import SimpleNamespace
import sys

# Add GenNet utils to path
sys.path.insert(0, str(Path("..").resolve()))
from GenNet_utils.Train_network import train_model

import matplotlib.pyplot as plt
plt.style.use('bmh')

np.random.seed(42)

## Configuration

In [9]:
# Paths
base_dir = Path('../data/processed/pure_grid_experiments')
h5_dir = Path('../data/processed/pure_simulation/h5_output')
results_dir = Path('../results/scrambled_topology_experiments')
results_dir.mkdir(parents=True, exist_ok=True)

# Load existing GenNet results for comparison
gennet_results = pd.read_csv('../results/pure_grid_results_relu.csv')

# Select representative experiments to test (to reduce computation time)
# Choose diverse scenarios: different N, P, alpha
test_experiments = [
    {'n_train': 15000, 'P': 10, 'h2': 0.6, 'alpha': 0},   # Large N, low P, additive
    {'n_train': 15000, 'P': 10, 'h2': 0.6, 'alpha': 1},   # Large N, low P, epistatic
    {'n_train': 15000, 'P': 50, 'h2': 0.6, 'alpha': 0},   # Large N, high P, additive
    {'n_train': 15000, 'P': 50, 'h2': 0.6, 'alpha': 1},   # Large N, high P, epistatic
]

print(f"Testing {len(test_experiments)} representative experiments:")
for exp in test_experiments:
    print(f"  N={exp['n_train']}, P={exp['P']}, h²={exp['h2']}, α={exp['alpha']}")

# Fixed parameters
n_snps = 10000
n_genes = 500

Testing 4 representative experiments:
  N=15000, P=10, h²=0.6, α=0
  N=15000, P=10, h²=0.6, α=1
  N=15000, P=50, h²=0.6, α=0
  N=15000, P=50, h²=0.6, α=1


## Topology Shuffling Functions

In [4]:
def load_topology(topology_file):
    """
    Load topology from CSV.
    
    Expected columns:
    - chr, layer0_node (SNP), layer0_name, layer1_node (Gene), layer1_name
    """
    topology = pd.read_csv(topology_file)
    print(f"Loaded topology: {len(topology)} connections")
    print(f"  SNPs: {topology['layer0_node'].nunique()}")
    print(f"  Genes: {topology['layer1_node'].nunique()}")
    return topology


def shuffle_topology(topology_df, scramble_fraction=1.0, seed=None):
    """
    Partially scramble SNP → Gene assignments with controlled proportion.

    Parameters:
    -----------
    topology_df : pd.DataFrame
        Original topology with SNP→Gene connections
    scramble_fraction : float (0.0 to 1.0)
        Proportion of SNPs to reassign to different genes
        - 0.0: No scrambling (return original)
        - 0.5: Scramble 50% of SNPs
        - 1.0: Full scrambling (all SNPs reassigned)
    seed : int, optional
        Random seed for reproducibility

    Returns:
    --------
    pd.DataFrame : Partially scrambled topology with same sparsity structure

    Algorithm (Simplified):
    -----------------------
    1. Randomly sample SNPs to scramble (scramble_fraction × n_snps)
    2. Keep non-scrambled SNPs in their original gene assignments
    3. Get topology rows for scrambled SNPs
    4. Randomly shuffle the SNP indices within those rows
    5. Combine kept and scrambled topologies
    """
    if seed is not None:
        np.random.seed(seed)

    # Special case: no scrambling
    if scramble_fraction == 0.0:
        print("\nNo scrambling (scramble_fraction=0.0)")
        print("  Returning original topology")
        return topology_df.copy()

    # Step 1: Get unique SNPs and sample which ones to scramble
    unique_snps = topology_df['layer0_node'].unique()
    n_snps_total = len(unique_snps)
    n_snps_to_scramble = int(n_snps_total * scramble_fraction)
    n_snps_to_keep = n_snps_total - n_snps_to_scramble

    # Randomly select SNPs to scramble
    snps_to_scramble = np.random.choice(unique_snps, size=n_snps_to_scramble, replace=False)
    snps_to_keep = np.setdiff1d(unique_snps, snps_to_scramble)

    # Step 2: Keep topology rows for non-scrambled SNPs
    kept_topology = topology_df[topology_df['layer0_node'].isin(snps_to_keep)].copy()

    # Step 3: Get topology rows for scrambled SNPs
    scrambled_rows = topology_df[topology_df['layer0_node'].isin(snps_to_scramble)].copy()

    # Step 4: Randomly shuffle the SNP indices within scrambled rows
    shuffled_snp_indices = np.random.permutation(scrambled_rows['layer0_node'].values)
    scrambled_rows['layer0_node'] = shuffled_snp_indices
    scrambled_rows['layer0_name'] = [f'SNP_{idx}' for idx in shuffled_snp_indices]

    # Step 5: Combine kept and scrambled topologies
    new_topology = pd.concat([kept_topology, scrambled_rows], ignore_index=True)

    # Verify structure preservation
    assert len(new_topology) == len(topology_df), "Connection count changed!"
    assert new_topology['layer0_node'].nunique() == n_snps_total, "SNP count changed!"
    assert new_topology['layer1_node'].nunique() == topology_df['layer1_node'].nunique(), "Gene count changed!"

    # Calculate realized overlap
    orig_connections = set(zip(topology_df['layer0_node'], topology_df['layer1_node']))
    new_connections = set(zip(new_topology['layer0_node'], new_topology['layer1_node']))

    overlap = orig_connections & new_connections
    overlap_pct = len(overlap) / len(orig_connections) * 100
    expected_overlap_pct = (1 - scramble_fraction) * 100

    print(f"\nPartial scrambling complete:")
    print(f"  Scramble fraction: {scramble_fraction:.1%}")
    print(f"  SNPs scrambled: {n_snps_to_scramble:,} / {n_snps_total:,}")
    print(f"  SNPs kept original: {n_snps_to_keep:,}")
    print(f"  Realized overlap: {overlap_pct:.1f}% (expected ~{expected_overlap_pct:.1f}%)")
    print(f"  Structure preserved: ✓")

    return new_topology


def calculate_topology_overlap(original, shuffled):
    """
    Calculate what fraction of SNP→Gene connections are preserved.
    """
    # Create connection tuples
    orig_connections = set(zip(original['layer0_node'], original['layer1_node']))
    shuf_connections = set(zip(shuffled['layer0_node'], shuffled['layer1_node']))
    
    # Calculate overlap
    overlap = orig_connections & shuf_connections
    overlap_pct = len(overlap) / len(orig_connections) * 100
    
    print(f"\nTopology overlap:")
    print(f"  Preserved connections: {len(overlap)} / {len(orig_connections)} ({overlap_pct:.1f}%)")
    print(f"  Expected for random: ~{100/n_genes:.1f}%")
    
    return overlap_pct


# Test the partial scrambling function
test_exp_id = f"exp_N{test_experiments[0]['n_train']}_P{test_experiments[0]['P']}_h2{test_experiments[0]['h2']}_alpha{test_experiments[0]['alpha']}"
test_topology_file = base_dir / test_exp_id / 'topology.csv'

print("="*70)
print("TESTING PARTIAL SCRAMBLING FUNCTION")
print("="*70)

if test_topology_file.exists():
    original_topo = load_topology(test_topology_file)
    
    # Test different scramble fractions
    for frac in [0.0, 0.25, 0.5, 0.75, 1.0]:
        print(f"\n--- Testing scramble_fraction={frac:.2f} ---")
        scrambled_topo = shuffle_topology(original_topo, scramble_fraction=frac, seed=123)
else:
    print(f"Test topology file not found: {test_topology_file}")
    print("Skipping function test")

TESTING PARTIAL SCRAMBLING FUNCTION
Loaded topology: 10000 connections
  SNPs: 10000
  Genes: 500

--- Testing scramble_fraction=0.00 ---

No scrambling (scramble_fraction=0.0)
  Returning original topology

--- Testing scramble_fraction=0.25 ---

Partial scrambling complete:
  Scramble fraction: 25.0%
  SNPs scrambled: 2,500 / 10,000
  SNPs kept original: 7,500
  Realized overlap: 75.1% (expected ~75.0%)
  Structure preserved: ✓

--- Testing scramble_fraction=0.50 ---

Partial scrambling complete:
  Scramble fraction: 50.0%
  SNPs scrambled: 5,000 / 10,000
  SNPs kept original: 5,000
  Realized overlap: 50.1% (expected ~50.0%)
  Structure preserved: ✓

--- Testing scramble_fraction=0.75 ---

Partial scrambling complete:
  Scramble fraction: 75.0%
  SNPs scrambled: 7,500 / 10,000
  SNPs kept original: 2,500
  Realized overlap: 25.2% (expected ~25.0%)
  Structure preserved: ✓

--- Testing scramble_fraction=1.00 ---

Partial scrambling complete:
  Scramble fraction: 100.0%
  SNPs scrambl

## FCNN Baseline Functions

In [5]:
# Import TensorFlow/Keras for FCNN
import tensorflow as tf
from tensorflow import keras
import tables
from sklearn.metrics import r2_score, mean_squared_error


def create_fcnn_same_shape(n_snps, n_hidden, l1_lambda=0.01, activation='relu'):
    """
    Create FCNN with same layer dimensions as GenNet.
    
    This has MANY more parameters than GenNet:
    Dense(500): 10000 × 500 + 500 = 5,000,500 params
    Dense(1): 500 × 1 + 1 = 501 params
    Total: ~5,001,001 params (vs GenNet's ~11,000)
    """
    model = keras.models.Sequential([
        keras.Input(shape=(n_snps,)),
        keras.layers.BatchNormalization(center=False, scale=False),
        keras.layers.Dense(n_hidden, activation=activation,
                          kernel_regularizer=keras.regularizers.l1(l1_lambda)),
        keras.layers.BatchNormalization(center=False, scale=False),
        keras.layers.Dense(1, activation='linear',
                          kernel_regularizer=keras.regularizers.l1(l1_lambda))
    ])
    
    return model

def train_fcnn(model, X_train, y_train, X_val, y_val, 
               lr=0.001, batch_size=32, epochs=150, patience=10, verbose=0):
    """
    Train FCNN model with early stopping.
    """
    # Compile model
    optimizer = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, 
                  loss='mean_squared_error',
                  metrics=['mean_squared_error'])
    
    # Callbacks
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True
    )
    
    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose
    )
    
    return history


def evaluate_fcnn(model, X_test, y_test):
    """
    Evaluate model and return R² and MSE.
    """
    y_pred = model.predict(X_test, verbose=0).flatten()
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    return r2, mse


def load_genotype_data(h5_path):
    """
    Load genotype matrix from HDF5 file.
    
    The genotype.h5 file contains:
    - 'data': array of shape (n_samples, n_snps)
    """
    with tables.open_file(str(h5_path), 'r') as h5:
        genotype_matrix = h5.root.data[:]
    
    print(f"Loaded genotype matrix: {genotype_matrix.shape}")
    return genotype_matrix


def load_experiment_data(exp_dir, genotype_matrix):
    """
    Load experiment data (phenotypes, train/val/test splits).
    
    subjects.csv contains:
    - patient_id: sample identifier
    - labels: phenotype values
    - genotype_row: row index in genotype matrix
    - set: 1=train, 2=val, 3=test
    """
    subjects_df = pd.read_csv(exp_dir / 'subjects.csv')
    
    # Split by set
    train_mask = subjects_df['set'] == 1
    val_mask = subjects_df['set'] == 2
    test_mask = subjects_df['set'] == 3
    
    # Get indices and labels
    train_idx = subjects_df[train_mask]['genotype_row'].values
    val_idx = subjects_df[val_mask]['genotype_row'].values
    test_idx = subjects_df[test_mask]['genotype_row'].values
    
    y_train = subjects_df[train_mask]['labels'].values
    y_val = subjects_df[val_mask]['labels'].values
    y_test = subjects_df[test_mask]['labels'].values
    
    # Get genotype data
    X_train = genotype_matrix[train_idx]
    X_val = genotype_matrix[val_idx]
    X_test = genotype_matrix[test_idx]
    
    return X_train, X_val, X_test, y_train, y_val, y_test

## Train Models with Scrambled Topology

In [6]:
import os

results_path = Path(os.path.dirname(os.getcwd())) / 'results'

# Configuration for gradient analysis
scramble_fractions = [0.0, 0.25, 0.5, 0.75, 1.0]  
n_seeds = 3  # Multiple seeds for error bars
l1_values = [0.01, 0.1, 1.0]  # L1 sweep for FCNN 

all_results = []
experiment_count = 0

# Load genotype matrix once (shared across all experiments)
print("="*70)
print("Loading shared genotype matrix...")
print("="*70)
genotype_matrix = load_genotype_data(h5_dir / 'genotype.h5')

print(f"Total GenNet experiments: {len(test_experiments)} × {len(scramble_fractions)} × {n_seeds} = {len(test_experiments) * len(scramble_fractions) * n_seeds}")
print(f"Total FCNN experiments: {len(test_experiments)}")


for config_idx, exp_config in enumerate(test_experiments):
    exp_id = f"exp_N{exp_config['n_train']}_P{exp_config['P']}_h2{exp_config['h2']}_alpha{exp_config['alpha']}"
    exp_dir = base_dir / exp_id
    
    if not exp_dir.exists():
        print(f"\n[CONFIG {config_idx+1}/{len(test_experiments)}] {exp_id} - SKIP (not found)")
        continue
    
    print(f"\n[CONFIG {config_idx+1}/{len(test_experiments)}] {exp_id}")
    print("="*60)
    
    # Load original topology
    original_topology = pd.read_csv(exp_dir / 'topology.csv')
    
    # Load experiment data for FCNN
    X_train, X_val, X_test, y_train, y_val, y_test = load_experiment_data(exp_dir, genotype_matrix)
    print(f"  Data loaded: Train={X_train.shape[0]}, Val={X_val.shape[0]}, Test={X_test.shape[0]}")
    
    # ============================================
    # FCNN Baseline (once per config)
    # ============================================
    print(f"\n  [FCNN Baseline] Training with L1 sweep...")
    best_val_r2 = -np.inf
    best_l1 = None
    fcnn_test_r2 = None
    fcnn_test_mse = None
    
    for l1_val in l1_values:
        tf.keras.backend.clear_session()
        
        fcnn_model = create_fcnn_same_shape(n_snps, n_genes, l1_lambda=l1_val, activation='relu')
        
        history = train_fcnn(
            fcnn_model, X_train, y_train, X_val, y_val,
            lr=0.001, batch_size=32, epochs=150, patience=10, verbose=0
        )
        
        test_r2, test_mse = evaluate_fcnn(fcnn_model, X_test, y_test)
        val_r2, val_mse = evaluate_fcnn(fcnn_model, X_val, y_val)
        
        print(f"    L1={l1_val}: Val R²={val_r2:.4f}, Test R²={test_r2:.4f}")
        
        if val_r2 > best_val_r2:
            best_val_r2 = val_r2
            best_l1 = l1_val
            fcnn_test_r2 = test_r2
            fcnn_test_mse = test_mse
    
    print(f"    Best L1={best_l1}: Test R²={fcnn_test_r2:.4f}")
    
    # Store FCNN result for all scramble fractions (topology-independent)
    for scramble_frac in scramble_fractions:
        all_results.append({
            'experiment_id': exp_id,
            'n_train': exp_config['n_train'],
            'P': exp_config['P'],
            'h2': exp_config['h2'],
            'alpha': exp_config['alpha'],
            'scramble_fraction': scramble_frac,
            'model': 'FCNN',
            'seed': None,
            'test_r2': fcnn_test_r2,
            'test_mse': fcnn_test_mse,
            'val_r2': best_val_r2,
            'best_l1': best_l1
        })
    
    # ============================================
    # GenNet with varying scramble fractions
    # ============================================
    for scramble_frac in scramble_fractions:
        for seed_idx in range(n_seeds):
            print(f"\n  [GenNet] scramble={scramble_frac:.2f}, seed={seed_idx+1}/{n_seeds}")
            
            # Generate scrambled topology
            scrambled_topology = shuffle_topology(
                original_topology,
                scramble_fraction=scramble_frac,
                seed=1000 + experiment_count
            )
            
            # Save to temp directory
            temp_dir = results_dir / f"temp_{exp_id}_scram{int(scramble_frac*100)}_seed{seed_idx}"
            temp_dir.mkdir(exist_ok=True, parents=True)
            
            # Copy files
            shutil.copy(exp_dir / 'subjects.csv', temp_dir / 'subjects.csv')
            scrambled_topology.to_csv(temp_dir / 'topology.csv', index=False)
            
            # Job ID for this run
            job_id = 50000 + experiment_count
            
            # Prepare training arguments
            args = SimpleNamespace(
                path=str(temp_dir) + '/',
                ID=job_id,
                genotype_path=str(h5_dir),
                problem_type='regression',
                regression=True,
                wpc=1,
                learning_rate=0.001,
                batch_size=32,
                epochs=150,
                workers=1,
                L1=0.01,
                L1_act=0.01,
                network_name='undefined',
                filters=2,
                mixed_precision=False,
                suffix='',
                out='undefined',
                mask_order=[],
                epoch_size=None,
                patience=10,
                resume=False,
                onehot=False,
                init_linear=False,
                improved_norm=False,
                verbose=0,
                activation_type='relu'
            )
            
            try:
                # Train model
                train_model(args)
                
                # Parse results
                result_dir = results_path / f'GenNet_experiment_{job_id}_'
                summary_file = result_dir / 'results_summary.txt'
                
                if summary_file.exists():
                    metrics = {}
                    with open(summary_file, 'r') as f:
                        for line in f:
                            if ':' in line:
                                key, value = line.strip().split(':', 1)
                                metrics[key] = value.strip()
                    
                    test_r2 = float(metrics.get('R2_test', 0))
                    test_mse = float(metrics.get('MSE test', 0))
                    val_r2 = float(metrics.get('R2_validation', 0))
                    val_mse = float(metrics.get('MSE validation', 0))
                    
                    # Store result
                    all_results.append({
                        'experiment_id': exp_id,
                        'n_train': exp_config['n_train'],
                        'P': exp_config['P'],
                        'h2': exp_config['h2'],
                        'alpha': exp_config['alpha'],
                        'scramble_fraction': scramble_frac,
                        'model': 'GenNet',
                        'seed': seed_idx,
                        'test_r2': test_r2,
                        'test_mse': test_mse,
                        'val_r2': val_r2,
                        'val_mse': val_mse
                    })
                    
                    print(f"    Test R²={test_r2:.4f}, Val R²={val_r2:.4f}")
                else:
                    print(f"    WARNING: Results file not found")
            
            except Exception as e:
                print(f"    ERROR: {e}")
                import traceback
                traceback.print_exc()
            
            experiment_count += 1

print("experiments complete.")

# Save results
df_results = pd.DataFrame(all_results)
df_results.to_csv(results_dir / 'scramble_gradient_with_fcnn.csv', index=False)
print(f"\nResults saved to: {results_dir / 'scramble_gradient_with_fcnn.csv'}")
print(f"Total rows: {len(df_results)}")
print(df_results.head(10))

Loading shared genotype matrix...
Loaded genotype matrix: (25000, 10000)
Total GenNet experiments: 4 × 5 × 3 = 60
Total FCNN experiments: 4

[CONFIG 1/4] exp_N15000_P10_h20.6_alpha0
  Data loaded: Train=15000, Val=3000, Test=3000

  [FCNN Baseline] Training with L1 sweep...
Metal device set to: Apple M4

systemMemory: 16.00 GB
maxCacheSize: 5.92 GB



2026-02-11 22:53:45.454325: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-11 22:53:45.456172: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-02-11 22:53:46.285060: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz
2026-02-11 22:53:46.582020: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2026-02-11 22:53:51.142795: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2026-02-11 22:55:02.087261: I tensorflow/core/grappler/o

    L1=0.01: Val R²=0.4363, Test R²=0.4362


2026-02-11 22:55:03.467655: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2026-02-11 22:55:08.025051: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.
2026-02-11 22:55:52.650190: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


    L1=0.1: Val R²=0.0012, Test R²=0.0012


...

467/469 [============================>.] - ETA: 0s - loss: 1.5780 - mse: 1.1179

2026-02-12 01:30:08.998785: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.



Epoch 1: val_loss improved from inf to 1.31933, saving model to /Users/jhu/Documents/broad/GenNet/results/GenNet_experiment_50059_/bestweights_job.h5
469/469 [==============================] - 6s 11ms/step - loss: 1.5774 - mse: 1.1177 - val_loss: 1.3193 - val_mse: 0.9731 - lr: 0.0010
0 left_in_epoch
Shuffeling epochs
Epoch 2/150
464/469 [============================>.] - ETA: 0s - loss: 1.2068 - mse: 0.9383
Epoch 2: val_loss improved from 1.31933 to 1.14004, saving model to /Users/jhu/Documents/broad/GenNet/results/GenNet_experiment_50059_/bestweights_job.h5
469/469 [==============================] - 5s 10ms/step - loss: 1.2069 - mse: 0.9392 - val_loss: 1.1400 - val_mse: 0.9445 - lr: 0.0010
0 left_in_epoch
Shuffeling epochs
Epoch 3/150
466/469 [============================>.] - ETA: 0s - loss: 1.0487 - mse: 0.8769
Epoch 3: val_loss did not improve from 1.14004
469/469 [==============================] - 5s 10ms/step - loss: 1.0496 - mse: 0.8779 - val_loss: 1.2351 - val_mse: 1.0818 - lr

2026-02-12 01:31:23.149002: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


Mean squared error = 0.9317875990672906
Explained variance = 0.05927375289855219
r2 = 0.055367495927676424
DEBUG: Subsampling 3000 points to 1000 for plotting to save memory
Analysis over the test set
DEBUG: Creating test set generator...
DEBUG: Test generator created. Batch size: 32, Test size: 3000
DEBUG: Running predictions...
DEBUG: Predictions complete. Shape: (3000, 1)
DEBUG: Loading test labels...
DEBUG: Test labels loaded. Shape: (3000, 1)
DEBUG: Evaluating regression performance...
Mean squared error = 0.9694836358133189
Explained variance = 0.0826027369194452
r2 = 0.08253004934251473
DEBUG: Subsampling 3000 points to 1000 for plotting to save memory
DEBUG: Evaluation complete
DEBUG: Saving predictions...
DEBUG: Saving figure...
DEBUG: Figure saved
DEBUG: Closing figures to free memory...
DEBUG: Figures closed
DEBUG: Starting to save results summary...
DEBUG: Creating pandas summary...
DEBUG: Saving summary CSV...
DEBUG: Summary CSV saved
DEBUG: Saving results summary text fil

## Save Results

In [7]:
# Save scrambled results
df_scrambled = pd.DataFrame(scrambled_results)
df_scrambled.to_csv(results_dir / 'scrambled_topology_results.csv', index=False)

print(f"Scrambled topology results saved: {len(df_scrambled)} experiments")
print(f"\nResults:")
df_scrambled

NameError: name 'scrambled_results' is not defined

## Comparison Analysis

In [ ]:
# Combine original and scrambled results
original_subset = gennet_results[
    gennet_results['experiment_id'].isin([f"exp_N{e['n_train']}_P{e['P']}_h2{e['h2']}_alpha{e['alpha']}" 
                                           for e in test_experiments])
].copy()

original_subset['topology'] = 'original'
df_scrambled['topology'] = 'scrambled'

# Combine
combined = pd.concat([
    original_subset[['experiment_id', 'n_train', 'P', 'h2', 'alpha', 'test_r2', 'topology']],
    df_scrambled[['experiment_id', 'n_train', 'P', 'h2', 'alpha', 'test_r2', 'topology']]
], ignore_index=True)

print("="*60)
print("COMPARISON: Original vs Scrambled Topology")
print("="*60)

# Calculate differences
print("\n--- Test R² by Topology ---")
summary = combined.groupby('topology')['test_r2'].agg(['mean', 'std', 'min', 'max'])
print(summary)

print("\n--- Pairwise Comparison ---")
for exp_config in test_experiments:
    exp_id = f"exp_N{exp_config['n_train']}_P{exp_config['P']}_h2{exp_config['h2']}_alpha{exp_config['alpha']}"
    
    orig = combined[(combined['experiment_id'] == exp_id) & (combined['topology'] == 'original')]['test_r2'].values
    scram = combined[(combined['experiment_id'] == exp_id) & (combined['topology'] == 'scrambled')]['test_r2'].values
    
    if len(orig) > 0 and len(scram) > 0:
        diff = orig[0] - scram[0]
        winner = 'Original' if diff > 0 else 'Scrambled'
        
        print(f"\n{exp_id}:")
        print(f"  Original:  R² = {orig[0]:.4f}")
        print(f"  Scrambled: R² = {scram[0]:.4f}")
        print(f"  Difference: {diff:+.4f} ({winner} wins)")

# Overall statistics
pivot = combined.pivot_table(index=['n_train', 'P', 'alpha'], columns='topology', values='test_r2')
pivot['difference'] = pivot['original'] - pivot['scrambled']

print("\n--- Summary Statistics ---")
print(f"Mean difference (original - scrambled): {pivot['difference'].mean():.4f}")
print(f"Original wins: {(pivot['difference'] > 0).sum()} / {len(pivot)} experiments")
print(f"Scrambled wins: {(pivot['difference'] < 0).sum()} / {len(pivot)} experiments")

NameError: name 'df_scrambled' is not defined

## Visualization

In [11]:
# Load results
df_results = pd.read_csv(results_dir / 'scramble_gradient_with_fcnn.csv')

# Create 2x2 grid (one subplot per configuration)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

config_names = {
    (10, 0): 'P=10, Additive (α=0)',
    (10, 1): 'P=10, Epistatic (α=1)',
    (50, 0): 'P=50, Additive (α=0)',
    (50, 1): 'P=50, Epistatic (α=1)'
}

for idx, (P_val, alpha_val) in enumerate([(10, 0), (10, 1), (50, 0), (50, 1)]):
    ax = axes[idx]
    
    # Filter data for this configuration
    config_data = df_results[
        (df_results['P'] == P_val) &
        (df_results['alpha'] == alpha_val)
    ]
    
    if len(config_data) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(config_names[(P_val, alpha_val)], fontsize=12, fontweight='bold')
        continue
    
    # GenNet: Calculate mean and std across seeds
    gennet_data = config_data[config_data['model'] == 'GenNet']
    gennet_summary = gennet_data.groupby('scramble_fraction')['test_r2'].agg(['mean', 'std', 'count'])
    
    x = gennet_summary.index.values
    y_mean = gennet_summary['mean'].values
    y_std = gennet_summary['std'].values
    
    # Plot GenNet with error bars
    ax.errorbar(x, y_mean, yerr=y_std, marker='o', linewidth=2.5, color='#2ca02c',
                label='GenNet', markersize=8, capsize=5, capthick=2)
    
    # FCNN baseline (flat line, same for all scramble fractions)
    fcnn_data = config_data[config_data['model'] == 'FCNN']
    if len(fcnn_data) > 0:
        fcnn_r2 = fcnn_data['test_r2'].iloc[0]
        ax.axhline(y=fcnn_r2, color='#d62728', linestyle='--', linewidth=2,
                   label='FCNN Baseline')
    
    # Formatting
    ax.set_xlabel('Scramble Fraction', fontsize=11, fontweight='bold')
    ax.set_ylabel('Test R²', fontsize=11, fontweight='bold')
    ax.set_title(config_names[(P_val, alpha_val)], fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    
    # Add crossover annotation if GenNet falls below FCNN
    if len(fcnn_data) > 0:
        crossover_idx = np.where(y_mean < fcnn_r2)[0]
        if len(crossover_idx) > 0:
            crossover_frac = x[crossover_idx[0]]
            ax.axvline(x=crossover_frac, color='red', linestyle=':', alpha=0.5)
            ax.text(crossover_frac, ax.get_ylim()[1]*0.95,
                    f'Crossover: {crossover_frac:.0%}',
                    ha='center', fontsize=8, 
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(results_dir / 'scramble_gradient_with_fcnn.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Plot saved to: {results_dir / 'scramble_gradient_with_fcnn.png'}")


✓ Plot saved to: ../results/scrambled_topology_experiments/scramble_gradient_with_fcnn.png


## Interpretation

### What the gradient analysis reveals:

**Key Questions Answered:**

1. **Is sparse topology always beneficial?**
   - If GenNet never falls below FCNN → Yes, sparse structure helps even when scrambled
   - If GenNet crosses below at high scrambling → No, correct topology is required
   - Look for crossover points in the plots

2. **How robust is GenNet to topology errors?**
   - Crossover at ~10% → Very sensitive, needs accurate annotations
   - Crossover at ~75% → Robust, tolerates significant annotation errors
   - No crossover → Always better than FCNN (regularization benefit)

3. **Performance degradation pattern:**
   - Linear slope → Gradual, predictable loss
   - Steep drop after threshold → Critical topology requirement

**Expected Patterns:**

**Additive scenarios (α=0):**
- GenNet may stay above FCNN even at 100% scrambling
- Sparse structure provides regularization benefit
- Dimensionality reduction (10K → 500) helps generalization

**Epistatic scenarios (α=1):**
- GenNet more likely to cross below FCNN at high scrambling
- Scrambling destroys within-gene interactions
- FCNN may perform better when topology is completely wrong

### Visualization Features:

- **Green line with error bands**: GenNet performance across scramble gradient (mean ± std over 3 seeds)
- **Red dashed line**: FCNN baseline (constant, topology-independent)
- **Green horizontal line**: Original GenNet (0% scrambled) reference
- **Red vertical dotted line**: Crossover point where GenNet falls below FCNN
- **Error bands**: Variation across random seeds

### Next Steps:

To properly test whether biological annotations help:
1. Use real genotype data with LD structure (e.g., 1000 Genomes)
2. Use real gene annotations (GENCODE, Ensembl)
3. Simulate phenotypes where causal variants are in real genes
4. Compare real annotations vs scrambled annotations

**Key Insight:** Pure synthetic data tests the methodology, but biological interpretation requires real gene annotations where the causal structure aligns with biological function.

In [ ]:
print("\n" + "="*70)
print("SCRAMBLED TOPOLOGY GRADIENT ANALYSIS COMPLETE")
print("="*70)
print(f"\nResults saved to: {results_dir}")
print(f"\nKey files:")
print(f"  - scramble_gradient_with_fcnn.csv (full results)")
print(f"  - scramble_gradient_with_fcnn.png (visualization)")
print(f"\nAnalysis Summary:")
print(f"  Scramble fractions tested: {scramble_fractions}")
print(f"  Seeds per fraction: {n_seeds}")
print(f"  Configurations: {len(test_experiments)}")
print(f"  Total experiments: {len(df_results)} rows")

# Quick summary statistics
print(f"\n--- Performance Summary ---")
for P_val in [10, 50]:
    for alpha_val in [0, 1.0]:
        config_data = df_results[
            (df_results['P'] == P_val) &
            (df_results['alpha'] == alpha_val)
        ]
        
        if len(config_data) == 0:
            continue
        
        # GenNet at 0% and 100% scrambling
        gennet_0 = config_data[(config_data['model'] == 'GenNet') & 
                               (config_data['scramble_fraction'] == 0.0)]['test_r2'].mean()
        gennet_100 = config_data[(config_data['model'] == 'GenNet') & 
                                 (config_data['scramble_fraction'] == 1.0)]['test_r2'].mean()
        
        # FCNN baseline
        fcnn_r2 = config_data[config_data['model'] == 'FCNN']['test_r2'].iloc[0]
        
        print(f"\nP={P_val}, α={alpha_val}:")
        print(f"  GenNet (0% scrambled):   R² = {gennet_0:.4f}")
        print(f"  GenNet (100% scrambled): R² = {gennet_100:.4f}")
        print(f"  FCNN baseline:           R² = {fcnn_r2:.4f}")
        print(f"  Performance loss: {(gennet_0 - gennet_100) / gennet_0 * 100:.1f}%")